# Project - AI resume analyzer & job match engine
1. upload resume
2. enter job description
3. analyze skills
4. calculate job match accuracy
5. Generate AI suggestions to improve the resume

In [19]:
# install dependencies
!pip install -q transformers sentence-transformers gradio pypdf scikit-learn

In [20]:
# import libraries
import gradio as gr
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader
import numpy as np

# Explaination
1. gradio --> create web interface
2. senetence transformer --> load hugging face embedding model
3. cosine similarity --> calculate similarity score
4. pdf redear --> read resume pdf
5. numpy --> numerical operations

In [21]:
# load hugging face model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Explaination
1. loads pretrained embedding model from hugging face
2. this model converts text to vector embeddings
3. this vector allows us to compare resume vs job description

In [22]:
# resume reader function
def read_pdf(file):
  pdf_reader = PdfReader(file)
  text = ""                                # reads uploaded pdf resume

  for page in pdf_reader.pages:             # initialize empty string
    text += page.extract_text()             # loop through every page

  return text

In [23]:
# skills extraction --> we check which of this exists in the resume
import pandas as pd
skills_db = pd.read_csv("tech_skills.csv")
skills_db = skills_db["skill"].tolist()

In [24]:
# skill matching function
def extract_skills(text):
  found_skills = []
  text = text.lower()                        # convert resume text to lowercase

  for skill in skills_db:
    if skill in text:
      found_skills.append(skill)

  return found_skills

In [25]:
# job match engine
def match_resume(resume_file, job_description):
  # reads the uploaded resume
  resume_text = read_pdf(resume_file)

  # convert resume & job description in embedding
  resume_embedding = model.encode([resume_text])
  job_embedding = model.encode([job_description])

  # similarity calculation b/w resume & job description
  similarity = cosine_similarity(resume_embedding, job_embedding)[0][0]

  # convert to percentage
  score_percentage = round(similarity * 100, 2)

  # extract resume & job skills
  resume_skills = extract_skills(resume_text)
  job_skills = extract_skills(job_description)

  # find missing skills
  missing_skills = list(set(job_skills) - set(resume_skills))

  suggestions = ""
  if len(missing_skills) > 0:
    suggestions = f"consider adding this skills: {', '.join(missing_skills)}"
  else:
    suggestions = "your resume already covers most job skills"

  return score_percentage, resume_skills, missing_skills, suggestions

In [26]:
interface = gr.Interface(
    fn=match_resume,                                         # main function executed when user click on submit
    inputs=[
        gr.File(label="upload Resume"),
        gr.Textbox(label="Job Description", lines=10)
    ],
    outputs=[
        gr.Number(label="Match Accuracy"),
        gr.Textbox(label="Skills found in resume"),
        gr.Textbox(label="Missing Skills"),
        gr.Textbox(label="Suggestions")
    ],
    title = "AI resume analyzer & job macth engine",
    description="Upload your resume and paste job description"
    )

In [27]:
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02574fcc5de30d9126.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
